# DKD Risk Prediction in Type 2 Diabetes
## 06B. Reader-Friendly Revision Results Summary

This notebook is a **read-only report** of the corrected All of Us analyses.

It does **not** fit models, construct phenotypes, select features, recalibrate probabilities,
or choose thresholds. It loads frozen outputs from Notebooks 04 and 06 and organizes them
into reviewer-facing and manuscript-facing summaries suitable for GitHub.


# 1. Setup


In [ ]:
import warnings
import gcsfs
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

warnings.filterwarnings("ignore")

migration_folder = "rw-migration-aou-rw-639fade3"
bucket = migration_folder

RESULT_DIR = f"{bucket}/aou_dkd/dkd_model_results_20260822"
SENS_DIR = f"{RESULT_DIR}/reviewer_sensitivity_20260822"
FINAL_DATE = "20260822"

fs = gcsfs.GCSFileSystem()

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 120)

def read_csv(path):
    if not fs.exists(path):
        raise FileNotFoundError(path)
    with fs.open(path, "r") as f:
        return pd.read_csv(f)

def save_csv(df, path):
    with fs.open(path, "w") as f:
        df.to_csv(f, index=False)
    print(f"Saved: {path}")

def num(x, digits=3):
    return "NA" if pd.isna(x) else f"{float(x):.{digits}f}"

def pct(x, digits=1):
    return "NA" if pd.isna(x) else f"{100*float(x):.{digits}f}%"

def ci(est, lo, hi, digits=3):
    if any(pd.isna(v) for v in [est, lo, hi]):
        return "NA"
    return f"{float(est):.{digits}f} ({float(lo):.{digits}f}–{float(hi):.{digits}f})"

def mdout(text):
    display(Markdown(text))

def show(df, index=False):
    display(Markdown(df.to_markdown(index=index)))

def one(df, **filters):
    x = df.copy()
    for col, value in filters.items():
        x = x.loc[x[col].eq(value)]
    if len(x) != 1:
        raise ValueError(f"Expected one row for {filters}; found {len(x)}")
    return x.iloc[0]


# 2. Load Frozen Outputs


In [ ]:
FILES_04 = {
    "performance": f"{RESULT_DIR}/test_performance_all_models_{FINAL_DATE}.csv",
    "paired": f"{RESULT_DIR}/paired_model_comparisons_{FINAL_DATE}.csv",
    "paired_sensitivity": f"{RESULT_DIR}/paired_sensitivity_differences_{FINAL_DATE}.csv",
    "policy": f"{RESULT_DIR}/fixed_specificity_sensitivity_80_85_90_95_{FINAL_DATE}.csv",
    "calibration": f"{RESULT_DIR}/test_calibration_metrics_{FINAL_DATE}.csv",
    "clinical": f"{RESULT_DIR}/clinical_baseline_vs_m2_1y_{FINAL_DATE}.csv",
    "algorithmic": f"{RESULT_DIR}/algorithmic_sensitivity_1y_M5_{FINAL_DATE}.csv",
    "algorithmic_delong": f"{RESULT_DIR}/algorithmic_sensitivity_delong_1y_M5_{FINAL_DATE}.csv",
    "reclassification": f"{RESULT_DIR}/case_reclassification_{FINAL_DATE}.csv",
}

FILES_06 = {
    "renal_audit": f"{SENS_DIR}/prelandmark_renal_abnormality_summary.csv",
    "documented_normal": f"{SENS_DIR}/documented_normal_prelandmark_renal_results.csv",
    "no_renal_abnormality": f"{SENS_DIR}/no_prelandmark_renal_abnormality_results.csv",
    "alternative_outcomes": f"{SENS_DIR}/alternative_outcome_model_results.csv",
    "alternative_outcome_counts": f"{SENS_DIR}/alternative_outcome_case_counts.csv",
    "longer_followup": f"{SENS_DIR}/longer_control_followup_results.csv",
    "fixed_horizon": f"{SENS_DIR}/fixed_horizon_model_results.csv",
    "fixed_horizon_counts": f"{SENS_DIR}/fixed_horizon_cohort_counts.csv",
    "missingness": f"{SENS_DIR}/missingness_by_incremental_layer_and_outcome.csv",
    "visit_density": f"{SENS_DIR}/visit_measurement_density_by_outcome.csv",
    "postlandmark_monitoring": f"{SENS_DIR}/postlandmark_kidney_monitoring_by_outcome.csv",
    "observation_density": f"{SENS_DIR}/performance_by_prelandmark_observation_density.csv",
    "prs_free": f"{SENS_DIR}/prs_free_nested_model_results.csv",
    "selection_washout": f"{SENS_DIR}/selection_bias_preindex_washout_continuous.csv",
    "selection_1y": f"{SENS_DIR}/selection_bias_source_vs_1y_included_continuous.csv",
    "independent_domains": f"{SENS_DIR}/independent_domain_model_results.csv",
    "independent_domain_delong": f"{SENS_DIR}/independent_domain_delong_bonferroni.csv",
    "m1_random": f"{SENS_DIR}/m1_vs_random_baselines.csv",
}

R04 = {name: read_csv(path) for name, path in FILES_04.items()}
R06 = {name: read_csv(path) for name, path in FILES_06.items()}

print(f"Loaded {len(R04)} Notebook 04 outputs and {len(R06)} Notebook 06 outputs.")


# 3. Executive Summary


In [ ]:
perf_1y = R04["performance"].loc[R04["performance"]["landmark"].eq("1y")].copy()

m1 = one(perf_1y, model_id="M1")
m2 = one(perf_1y, model_id="M2")
m3 = one(perf_1y, model_id="M3")
m4 = one(perf_1y, model_id="M4")
m5 = one(perf_1y, model_id="M5")

clinical = one(R04["clinical"], model="Clinical LR")

doc_m1 = one(R06["documented_normal"], model_id="M1")
doc_m2 = one(R06["documented_normal"], model_id="M2")

fixed5 = R06["fixed_horizon"].loc[
    R06["fixed_horizon"]["scenario"].eq("fixed_5y_post_landmark_ehr")
]
fixed5_m1 = one(fixed5, model_id="M1")
fixed5_m2 = one(fixed5, model_id="M2")

prs_m5 = one(R06["prs_free"], model_id="M5")

mdout(f"""
### Main corrected one-year result

The held-out TEST cohort contained **{int(m1['n_test']):,} participants**,
including **{int(m1['n_cases']):,} DKD cases**.

The largest information gain remained **M1→M2**, with AUROC increasing from
**{num(m1['AUROC'])}** to **{num(m2['AUROC'])}**
(Δ = **{m2['AUROC'] - m1['AUROC']:+.3f}**).
M4 and M5 reached AUROCs of **{num(m4['AUROC'])}** and **{num(m5['AUROC'])}**.

At the primary fixed-specificity policy, sensitivity increased from
**{pct(m1['Sensitivity'])}** for M1 to **{pct(m2['Sensitivity'])}** for M2.
M3 modestly improved global discrimination but had lower sensitivity than M2 at the
90%-specificity operating point.

### Reviewer-critical robustness checks

- **Renal predictor–outcome overlap:** documented-normal pre-landmark renal subgroup:
  **{num(doc_m1['AUROC'])} → {num(doc_m2['AUROC'])}**
  (Δ = **{doc_m2['AUROC'] - doc_m1['AUROC']:+.3f}**).
- **Fixed 5-year post-landmark horizon:** **{num(fixed5_m1['AUROC'])} → {num(fixed5_m2['AUROC'])}**
  (Δ = **{fixed5_m2['AUROC'] - fixed5_m1['AUROC']:+.3f}**).
- **Four-variable clinical benchmark:** AUROC **{num(clinical['AUROC'])}** versus
  M2 AUROC **{num(m2['AUROC'])}**.
- **PRS-free mature model:** M5 AUROC **{num(prs_m5['AUROC'])}**.

**Overall:** the principal M1→M2 laboratory-information gain persists across the major
reviewer-requested sensitivity analyses.
""")


# 4. Primary One-Year Performance


In [ ]:
x = perf_1y.copy()

table = pd.DataFrame({
    "Model": x["model_id"],
    "TEST N": x["n_test"].astype(int),
    "Cases": x["n_cases"].astype(int),
    "AUROC (95% CI)": [ci(a, lo, hi) for a, lo, hi in zip(x["AUROC"], x["AUROC_ci_low"], x["AUROC_ci_high"])],
    "AUPRC (95% CI)": [ci(a, lo, hi) for a, lo, hi in zip(x["AUPRC"], x["AUPRC_ci_low"], x["AUPRC_ci_high"])],
    "Sensitivity": x["Sensitivity"].map(pct),
    "Specificity": x["Specificity"].map(pct),
    "Brier": x["Brier"].map(num),
    "CITL": x["CITL"].map(num),
    "Calibration slope": x["Calibration_slope"].map(num),
    "CAL threshold": x["threshold_cal"].map(lambda v: num(v, 4)),
})
show(table)


In [ ]:
paired_1y = R04["paired"].loc[R04["paired"]["landmark"].eq("1y")].copy()

table = pd.DataFrame({
    "Transition": paired_1y["transition"],
    "ΔAUROC (95% CI)": [
        ci(a, lo, hi)
        for a, lo, hi in zip(
            paired_1y["delta_auroc"],
            paired_1y["delta_auroc_ci_low"],
            paired_1y["delta_auroc_ci_high"],
        )
    ],
    "DeLong p": paired_1y["delong_p"],
    "ΔAUPRC (95% CI)": [
        ci(a, lo, hi)
        for a, lo, hi in zip(
            paired_1y["delta_auprc"],
            paired_1y["delta_auprc_ci_low"],
            paired_1y["delta_auprc_ci_high"],
        )
    ],
    "McNemar p": paired_1y["mcnemar_p"],
})
show(table)

d12 = one(paired_1y, transition="M1 -> M2")
d23 = one(paired_1y, transition="M2 -> M3")
d34 = one(paired_1y, transition="M3 -> M4")
d45 = one(paired_1y, transition="M4 -> M5")

mdout(f"""
### Interpretation

- **M1→M2:** dominant improvement, ΔAUROC **{d12['delta_auroc']:+.3f}**.
- **M2→M3:** small global discrimination gain, ΔAUROC **{d23['delta_auroc']:+.3f}**.
- **M3→M4:** additional improvement, ΔAUROC **{d34['delta_auroc']:+.3f}**.
- **M4→M5:** globally near-neutral, ΔAUROC **{d45['delta_auroc']:+.3f}**.
""")


# 5. Fixed-Specificity Policy Robustness


In [ ]:
policy = R04["policy"].loc[
    R04["policy"]["landmark"].eq("1y")
    & R04["policy"]["analysis_group"].eq("primary_layered")
].copy()

table = pd.DataFrame({
    "Model": policy["model_id"],
    "Target specificity": policy["target_specificity"].map(pct),
    "Achieved CAL specificity": policy["cal_specificity"].map(pct),
    "CAL threshold": policy["threshold_cal"].map(lambda x: num(x, 4)),
    "TEST sensitivity": policy["Sensitivity"].map(pct),
    "TEST specificity": policy["Specificity"].map(pct),
})
show(table)

mdout("""
**Reading note.** Isotonic calibration produces discrete probability levels. Multiple
target-specificity values can therefore map to the same achievable threshold.
""")


# 6. Clinical Benchmark and Algorithmic Sensitivity


In [ ]:
clinical_table = R04["clinical"][
    ["model", "N", "cases", "AUROC", "AUPRC", "Sensitivity", "Specificity", "Brier"]
].copy()

for col in ["AUROC", "AUPRC", "Brier"]:
    clinical_table[col] = clinical_table[col].map(num)
for col in ["Sensitivity", "Specificity"]:
    clinical_table[col] = clinical_table[col].map(pct)

show(clinical_table)

clinical_lr = one(R04["clinical"], model="Clinical LR")
m2_xgb = one(R04["clinical"], model="M2 XGBoost")

mdout(f"""
The age + eGFR + UACR + HbA1c logistic benchmark had AUROC
**{num(clinical_lr['AUROC'])}**, compared with **{num(m2_xgb['AUROC'])}** for M2 XGBoost.
""")


In [ ]:
algorithm_table = R04["algorithmic"][
    ["model", "N", "cases", "AUROC", "AUPRC", "Sensitivity", "Specificity", "Brier"]
].copy()

for col in ["AUROC", "AUPRC", "Brier"]:
    algorithm_table[col] = algorithm_table[col].map(num)
for col in ["Sensitivity", "Specificity"]:
    algorithm_table[col] = algorithm_table[col].map(pct)

show(algorithm_table)
show(R04["algorithmic_delong"])


# 7. Renal Predictor–Outcome Circularity


In [ ]:
renal["no_prelandmark_renal_measurement"] = (
    1 - renal["has_prelandmark_renal_measurement"]
)

table = pd.DataFrame({
    "Future DKD": renal["dkd_binary"],
    "N": renal["n"].astype(int),
    "Any single pre-landmark renal abnormality": renal[
        "any_prelandmark_renal_abnormality"
    ].map(pct),
    "Documented normal pre-landmark renal measurements": renal[
        "documented_normal_prelandmark_renal"
    ].map(pct),
    "No pre-landmark renal measurement": renal[
        "no_prelandmark_renal_measurement"
    ].map(pct),
})
show(table)


In [ ]:
def scenario_summary(df, title):
    x = df.copy()
    table = pd.DataFrame({
        "Model": x["model_id"],
        "TEST N": x["n_test"].astype(int),
        "Cases": x["test_cases"].astype(int),
        "AUROC": x["AUROC"].map(num),
        "AUPRC": x["AUPRC"].map(num),
        "Sensitivity": x["Sensitivity"].map(pct),
        "Specificity": x["Specificity"].map(pct),
    })
    mdout(f"### {title}")
    show(table)

scenario_summary(R06["documented_normal"], "Documented normal pre-landmark renal measurements")
scenario_summary(R06["no_renal_abnormality"], "No observed pre-landmark renal abnormality")

doc1 = one(R06["documented_normal"], model_id="M1")
doc2 = one(R06["documented_normal"], model_id="M2")
nr1 = one(R06["no_renal_abnormality"], model_id="M1")
nr2 = one(R06["no_renal_abnormality"], model_id="M2")

mdout(f"""
### Interpretation

- documented-normal renal subgroup:
  **{doc1['AUROC']:.3f} → {doc2['AUROC']:.3f}**
  (Δ = **{doc2['AUROC'] - doc1['AUROC']:+.3f}**);
- excluding anyone with any observed pre-landmark eGFR <60 or UACR ≥30:
  **{nr1['AUROC']:.3f} → {nr2['AUROC']:.3f}**
  (Δ = **{nr2['AUROC'] - nr1['AUROC']:+.3f}**).

""")


# 8. Alternative DKD Outcome Definitions


In [ ]:
alt = R06["alternative_outcomes"].copy()
display_models = ["M1", "M2", "M4", "M5"]
x = alt.loc[alt["model_id"].isin(display_models)].copy()

table = pd.DataFrame({
    "Outcome": x["scenario"],
    "Model": x["model_id"],
    "TEST N": x["n_test"].astype(int),
    "Cases": x["test_cases"].astype(int),
    "AUROC": x["AUROC"].map(num),
    "AUPRC": x["AUPRC"].map(num),
})
show(table)

lines = []
for outcome in ["egfr_defined_dkd", "uacr_defined_dkd", "both_egfr_and_uacr_dkd", "dkd_icd_only"]:
    z = alt.loc[alt["scenario"].eq(outcome)]
    r1 = one(z, model_id="M1")
    r2 = one(z, model_id="M2")
    lines.append(
        f"- **{outcome}:** {r1['AUROC']:.3f} → {r2['AUROC']:.3f} "
        f"(Δ = {r2['AUROC'] - r1['AUROC']:+.3f})."
    )

mdout(
    "### M1→M2 by outcome definition\n\n"
    + "\n".join(lines)
    + "\n\nThe principal M1→M2 signal persists across component outcomes and a "
      "DKD-specific diagnosis-code-only endpoint."
)


# 9. Follow-up Duration and Fixed Prediction Horizons


In [ ]:
follow = R06["longer_followup"].copy()
table = pd.DataFrame({
    "Scenario": follow["scenario"],
    "Model": follow["model_id"],
    "TEST N": follow["n_test"].astype(int),
    "Cases": follow["test_cases"].astype(int),
    "AUROC": follow["AUROC"].map(num),
    "AUPRC": follow["AUPRC"].map(num),
    "Sensitivity": follow["Sensitivity"].map(pct),
    "Specificity": follow["Specificity"].map(pct),
})
show(table)


In [ ]:
show(
    R06["fixed_horizon_counts"].rename(
        columns={"scenario": "Scenario", "n": "N", "cases": "Cases", "controls": "Controls"}
    )
)

h = R06["fixed_horizon"].copy()
table = pd.DataFrame({
    "Scenario": h["scenario"],
    "Model": h["model_id"],
    "TEST N": h["n_test"].astype(int),
    "Cases": h["test_cases"].astype(int),
    "AUROC": h["AUROC"].map(num),
    "AUPRC": h["AUPRC"].map(num),
    "Sensitivity": h["Sensitivity"].map(pct),
    "Specificity": h["Specificity"].map(pct),
})
show(table)

for years in [3, 5]:
    z = h.loc[h["scenario"].eq(f"fixed_{years}y_post_landmark_ehr")]
    r1 = one(z, model_id="M1")
    r2 = one(z, model_id="M2")
    mdout(
        f"**Fixed {years}-year horizon:** M1→M2 AUROC "
        f"**{r1['AUROC']:.3f} → {r2['AUROC']:.3f}** "
        f"(Δ = **{r2['AUROC'] - r1['AUROC']:+.3f}**)."
    )


**Interpretation.** 

- The pattern of model performance was preserved when controls were required to have longer follow-up and when DKD was defined within fixed 3-year and 5-year windows after the one-year landmark. This makes it less likely that the primary results were driven mainly by unequal follow-up time between cases and controls.

- In a secondary analysis, participants who died before the end of the 3- or 5-year prediction window without first developing DKD were excluded rather than counted as controls. Results were essentially unchanged. This analysis was intended only to test whether early death was materially affecting the findings; it was not a formal competing-risk survival analysis.


# 10. Missingness, Visits, and Observation Density


In [ ]:
miss = R06["missingness"].copy()

table = pd.DataFrame({
    "Information layer": miss["domain"],
    "Future DKD": miss["outcome"],
    "N": miss["n"].astype(int),
    "Features in layer": miss["n_features"].astype(int),
    "Overall feature-value missingness": miss["cell_missing_fraction"].map(pct),
    "Mean participant-level missingness": miss[
        "mean_participant_missing_fraction"
    ].map(pct),
    "Participants with no observed features in layer": miss[
        "participants_all_features_missing_fraction"
    ].map(pct),
})

show(table)

In [ ]:
visit = R06["visit_density"].copy()
table = pd.DataFrame({
    "Future DKD": visit["dkd_binary"],
    "Mean pre-landmark visits": visit["prelandmark_visit_count_mean"].map(lambda x: num(x, 1)),
    "Median pre-landmark visits": visit["prelandmark_visit_count_median"].map(lambda x: num(x, 1)),
    "Mean observed general domains": visit["obs_general_any_mean"].map(lambda x: num(x, 1)),
    "Mean general measurement count": visit["obs_general_count_mean"].map(lambda x: num(x, 1)),
    "Median general measurement count": visit["obs_general_count_median"].map(lambda x: num(x, 1)),
})
show(table)

mdout("""

**Interpretation.** Participants who later developed DKD had more pre-landmark healthcare encounters than controls (mean 28.2 vs 24.0 visits) but fewer recorded measurements across the general clinical domains used to characterize observability (mean 94.6 vs 105.1 measurements; 5.0 vs 5.4 observed domains). Thus, visit frequency and availability of predictive clinical information were not equivalent: greater healthcare contact did not necessarily result in denser measurement of the physiologic domains used by the models.
""")


In [ ]:
density = R06["observation_density"].copy()
table = pd.DataFrame({
    "Model": density["model_id"],
    "Observation density": density["observation_density"],
    "N": density["n"].astype(int),
    "Cases": density["cases"].astype(int),
    "Case rate": density["case_rate"].map(pct),
    "AUROC": density["AUROC"].map(num),
    "AUPRC": density["AUPRC"].map(num),
    "Sensitivity": density["Sensitivity"].map(pct),
    "Specificity": density["Specificity"].map(pct),
})
show(table)

for model_id in ["M4", "M5"]:
    z = density.loc[density["model_id"].eq(model_id)]
    low = one(z, observation_density="Low")
    high = one(z, observation_density="High")
    mdout(
        f"**{model_id}:** AUROC increased from **{low['AUROC']:.3f}** in the low-observation "
        f"group to **{high['AUROC']:.3f}** in the high-observation group."
    )


# 11. Post-Landmark Kidney Monitoring


In [ ]:
show(R06["postlandmark_monitoring"])

mdout("""
**Interpretation.** Post-landmark kidney-testing counts are descriptive and should not be
interpreted as directly comparable monitoring rates between cases and controls. For cases,
the observation window ends when DKD occurs, whereas for controls it extends to the end of
available EHR follow-up. This creates different amounts of time in which kidney tests can
accumulate. We therefore use these counts only to describe post-landmark testing patterns,
not to infer differential surveillance intensity. Ascertainment bias is addressed more
directly by the DKD diagnosis-code-only outcome analysis and by analyses stratified by
pre-landmark observation density.
""")

# 12. PRS-Free Models


In [ ]:
prs = R06["prs_free"].copy()
table = pd.DataFrame({
    "Model": prs["model_id"],
    "TEST N": prs["n_test"].astype(int),
    "Cases": prs["test_cases"].astype(int),
    "AUROC": prs["AUROC"].map(num),
    "AUPRC": prs["AUPRC"].map(num),
    "Sensitivity": prs["Sensitivity"].map(pct),
    "Specificity": prs["Specificity"].map(pct),
})
show(table)

primary_m5 = one(perf_1y, model_id="M5")
prsfree_m5 = one(prs, model_id="M5")

mdout(f"""
The PRS-free M5 model retained an AUROC of **{prsfree_m5['AUROC']:.3f}**, compared with
**{primary_m5['AUROC']:.3f}** for the primary M5 model. This indicates that the predictive
performance of the full EHR-based model was largely preserved after removing PRS features.
""")

# 13. Independent Feature-Domain Models


In [ ]:
domains = R06["independent_domains"].copy()
table = pd.DataFrame({
    "Domain": domains["domain"],
    "Features": domains["n_features"].astype(int),
    "AUROC": domains["AUROC"].map(num),
    "AUPRC": domains["AUPRC"].map(num),
    "Brier": domains["Brier"].map(num),
    "Sensitivity": domains["Sensitivity"].map(pct),
    "Specificity": domains["Specificity"].map(pct),
})
show(table)
show(R06["independent_domain_delong"])

mdout("""
 
Snapshot and longitudinal features carry substantial standalone predictive information,
whereas medication and social/context domains are much weaker in isolation. The all-features
M5 model still exceeds every individual domain.
""")


# 14. M1 Versus Random Performance


In [ ]:
show(R06["m1_random"])
r = R06["m1_random"].iloc[0]

mdout(f"""
 
**Interpretation.** Reviewer 7 requested confirmation that M1 performed meaningfully above
random prediction. For AUROC, chance discrimination corresponds to 0.50. M1 achieved an
AUROC of **{r['AUROC']:.3f}**, which was significantly greater than 0.50
(p = **{r['AUROC_p_vs_0_5']:.2e}**).

For precision-recall performance, the random baseline is the outcome prevalence rather than
0.50. DKD prevalence in the TEST cohort was
**{r['AUPRC_random_baseline_prevalence']:.3f}**, compared with an M1 AUPRC of
**{r['AUPRC']:.3f}**. Thus, M1 performed above random expectation for both discrimination
metrics.
""")


# 15. Selection-Bias / Included-versus-Excluded Summaries


In [ ]:
mdout("### Effect of the pre-T2D observation requirement")
show(R06["selection_washout"])

mdout("### Initial eligible T2D cohort versus 1-year landmark cohort")
show(R06["selection_1y"])

mdout("""
  
**Interpretation.** These comparisons show whether the eligibility rules changed the
characteristics of the population included in the analysis.

The first table examines the effect of requiring sufficient EHR history before the first
documented T2D date. This requirement can exclude people who first entered the health
system close to the time their T2D was documented.

The second table compares the initial eligible T2D cohort with the participants who
remained eligible at the 1-year landmark after applying the landmark and follow-up
requirements. Differences between these groups indicate that the 1-year analysis may
represent a somewhat more selected population than the broader eligible T2D cohort.

These comparisons are descriptive. They show where selection may have occurred but do not
statistically correct for selection bias.
""")